# Experimentación de Modelado y Ensambles

En esta sección tomarán los aprendizajes del EDA para construir pipelines sólidos (Scikit-Learn). Luego experimentarán con varios algoritmos avanzados priorizados en árboles y ensambles.

## Pasos Clave
1. **Split Temporal**: Asegurar que Train (2015-2023), Validación (2024), Test (2025).
2. **Pipelines**: Creación ágil del Feature Engineering (e.j., escalado, imputación, variables dummy / target encoding).
3. **Entrenamiento**: Evaluación base usando Regresión Lineal, Voting, y comparar Bagging vs Pasting.
4. **Boosting**: Implementar CV y GridSearch para AdaBoost, GBDT, XGBoost, LightGBM, CatBoost.
5. **Selección**: Reporte final de RMSE/MAE en Train/Val de todos para encontrar al ganador. Solo con el ganador evaluar en Test (2025).

### 1. Carga de los Datos Limpios (o Query Directa)

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

# Añadir el root del proyecto al path para poder importar módulos de src
sys.path.append(os.path.abspath('..'))

from src.models.train_model import load_query_as_dataframe
from src.features.build_features import preprocess_data, get_feature_pipeline, TARGET
from src.models.evaluate_model import regression_metrics

# Queries usando SAMPLE para no saturar memoria en el notebook
TRAIN_QUERY = "SELECT * FROM ANALYTICS.REFINED.TRAIN_SET SAMPLE (5) LIMIT 100000"
VAL_QUERY = "SELECT * FROM ANALYTICS.REFINED.VAL_SET SAMPLE (10) LIMIT 20000"
TEST_QUERY = "SELECT * FROM ANALYTICS.REFINED.TEST_SET SAMPLE (10) LIMIT 20000"

print("Cargando datos...")
train_raw = load_query_as_dataframe(TRAIN_QUERY)
val_raw = load_query_as_dataframe(VAL_QUERY)
test_raw = load_query_as_dataframe(TEST_QUERY)
print(f"Train shape: {train_raw.shape}, Val shape: {val_raw.shape}, Test shape: {test_raw.shape}")

### 2. Separación Temporal (No Aleatoria)
El split ya lo hicimos en Snowflake, pero aquí aplicamos la limpieza de variables que causan Data Leakage y separamos X e Y.

In [ ]:
def prepare_data(df):
    df_clean = preprocess_data(df)
    X = df_clean.drop(columns=[TARGET])
    y = df_clean[TARGET]
    return X, y

X_train, y_train = prepare_data(train_raw)
X_val, y_val = prepare_data(val_raw)
X_test, y_test = prepare_data(test_raw)

print("Feature shape (Train):", X_train.shape)

### 3. Pipeline de Preprocesamiento
Usa `ColumnTransformer` o pipelines iterativos para tratar variables numéricas y categóricas.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Utilizamos el pipeline ya definido en src/features/build_features.py
preprocessor = get_feature_pipeline(X_train)
preprocessor

### 4. Entrenamiento Fase 1: Modelos Base (Regresión, Bagging, Pasting, Voting)
Evaluamos modelos más simples para establecer un punto de referencia antes de pasar a los algoritmos pesados.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import BaggingRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor
import time

base_models = {
    'LinearRegression': LinearRegression(),
    'Bagging': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5), n_estimators=20, random_state=42),
    'Pasting': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=5), n_estimators=20, bootstrap=False, random_state=42),
}
# Creamos un VotingRegressor combinando los anteriores
base_models['Voting_Ensemble'] = VotingRegressor(estimators=[
    ('lr', base_models['LinearRegression']),
    ('bagging', base_models['Bagging'])
])

results = {}
trained_pipelines = {}

for name, model in base_models.items():
    print(f"Entrenando {name}...")
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    
    start = time.time()
    pipeline.fit(X_train, y_train)
    t = time.time() - start
    
    y_pred_val = pipeline.predict(X_val)
    metrics = regression_metrics(y_val, y_pred_val)
    
    print(f"{name} - Tiempo: {t:.2f}s | RMSE Val: {metrics['rmse']:.4f}")
    results[name] = metrics
    trained_pipelines[name] = pipeline


### 5. Entrenamiento Fase 2: Boostings con Búsqueda de Hiperparámetros (CV)
Aplicamos validación cruzada (CV) a los modelos de Boosting. Para evitar que el notebook tarde horas, usamos `RandomizedSearchCV` con iteraciones cortas.

In [ ]:
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV

def create_pipeline(model):
    return Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

# Diccionario con Modelos y sus grillas de parámetros a buscar
boostings = {
    'AdaBoost': (AdaBoostRegressor(random_state=42), 
                 {'model__n_estimators': [50, 100], 'model__learning_rate': [0.05, 0.1]}),
    'GBDT': (GradientBoostingRegressor(random_state=42), 
             {'model__n_estimators': [50, 100], 'model__max_depth': [3, 5]}),
    'XGBoost': (XGBRegressor(random_state=42, n_jobs=-1), 
                {'model__n_estimators': [50, 100], 'model__max_depth': [3, 6]}),
    'LightGBM': (LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1), 
                 {'model__n_estimators': [50, 100], 'model__num_leaves': [31, 50]}),
    'CatBoost': (CatBoostRegressor(random_state=42, verbose=False), 
                 {'model__iterations': [50, 100], 'model__depth': [4, 6]})
}

for name, (model, params) in boostings.items():
    print(f"\nBuscando hiperparámetros para {name}...")
    pipeline = create_pipeline(model)
    
    # n_iter=2 y cv=2 para demostración rápida en notebook
    search = RandomizedSearchCV(pipeline, param_distributions=params, n_iter=2, cv=2, 
                                scoring='neg_root_mean_squared_error', random_state=42, n_jobs=1)
    
    start = time.time()
    search.fit(X_train, y_train)
    t = time.time() - start
    
    best_pipe = search.best_estimator_
    y_pred_val = best_pipe.predict(X_val)
    metrics = regression_metrics(y_val, y_pred_val)
    
    print(f"{name} - Mejor RMSE en CV: {-search.best_score_:.4f} | RMSE Val Final: {metrics['rmse']:.4f} | Tiempo: {t:.2f}s")
    results[name] = metrics
    trained_pipelines[name] = best_pipe


### 6. Selección y Evaluación Final
Comparamos los 9 modelos, escogemos al campeón por RMSE de Validación y lo evaluamos en el Set de Test de 2025.

In [ ]:
import matplotlib.pyplot as plt

print("--- RANKING FINAL DE VALIDACIÓN (RMSE) ---")
for k, v in sorted(results.items(), key=lambda x: x[1]['rmse']):
    print(f"{k}: {v['rmse']:.4f}")

best_model_name = min(results, key=lambda x: results[x]['rmse'])
best_pipeline = trained_pipelines[best_model_name]

print(f"\n🏆 EL GANADOR ABSOLUTO ES: {best_model_name}")

print("\n--- EVALUACIÓN EN TEST SET (2025) ---")
y_pred_test = best_pipeline.predict(X_test)
test_metrics = regression_metrics(y_test, y_pred_test)
print(f"Métricas en Test: {test_metrics}")

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_test, y_test - y_pred_test, alpha=0.3, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Valores Predichos')
plt.ylabel('Residuos (Real - Predicho)')
plt.title(f'Análisis de Residuos en Test - {best_model_name}')
plt.show()


### 7. Exportación del Modelo Ganador

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
model_path = "../models/best_experiment_model.pkl"
joblib.dump(best_pipeline, model_path)
print(f"Modelo empaquetado y guardado en: {model_path}")